In [26]:
import sys
print(sys.executable)

C:\Users\aryas\llm_project\llm_env\Scripts\python.exe


In [27]:
import langchain
print(langchain.__version__)

1.3.4


## Import Libraries

In [87]:
import os
from dotenv import load_dotenv
from youtube_transcript_api import (YouTubeTranscriptApi, TranscriptsDisabled)
from huggingface_hub import InferenceClient
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEndpoint
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [88]:
load_dotenv(".env")
HF_TOKEN = os.getenv("HF_TOKEN")

In [89]:
print(HF_TOKEN[:10])

hf_XOnjOxY


In [127]:
import youtube_transcript_api
print(dir(youtube_transcript_api))

['AgeRestricted', 'CookieError', 'CookieInvalid', 'CookiePathInvalid', 'CouldNotRetrieveTranscript', 'FailedToCreateConsentCookie', 'FetchedTranscript', 'FetchedTranscriptSnippet', 'InvalidVideoId', 'IpBlocked', 'NoTranscriptFound', 'NotTranslatable', 'PoTokenRequired', 'RequestBlocked', 'Transcript', 'TranscriptList', 'TranscriptsDisabled', 'TranslationLanguageNotAvailable', 'VideoUnavailable', 'VideoUnplayable', 'YouTubeDataUnparsable', 'YouTubeRequestFailed', 'YouTubeTranscriptApi', 'YouTubeTranscriptApiException', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_api', '_errors', '_settings', '_transcripts', 'proxies']


In [128]:
print(dir(YouTubeTranscriptApi))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'fetch', 'list']


## Step 1a - Indexing (Document Ingestion)

In [129]:
video_id = "Gfr50f6ZBvo"
ytt_api = YouTubeTranscriptApi()
transcript_data = ytt_api.fetch(video_id, languages=["en"])

print(type(transcript_data))

<class 'youtube_transcript_api._transcripts.FetchedTranscript'>


In [130]:
print(transcript_data[0])

FetchedTranscriptSnippet(text='the following is a conversation with', start=0.08, duration=3.44)


In [131]:
transcript = " ".join(snippet.text for snippet in transcript_data)
print(transcript[:1000])

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [132]:
print(len(transcript))

133836


## Step 1b - Indexing (Text Splitting)

In [133]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [134]:
print("Total Chunks:", len(chunks))

Total Chunks: 168


In [135]:
chunks[0]

Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inter

## Step 1c - Indexing (Embedding Generation)

In [136]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
print("Embedding Model Loaded")

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1983.29it/s]


Embedding Model Loaded


## Step 1d - Indexing (Storing in Vector Store)

In [137]:
vector_store = FAISS.from_documents(chunks, embeddings)

In [138]:
vector_store.index_to_docstore_id

{0: '10eb0324-49bc-467a-a0c2-0e3b3fa229da',
 1: 'b4e24499-7ee4-479a-93fe-8a92975a1192',
 2: '7f481dba-1a6d-4744-8114-03a499e4edad',
 3: '2eb8703d-ed64-42fa-8634-36f550061db5',
 4: '8b33a3c8-d724-4c8b-b5eb-8973ddcd0289',
 5: '095f75ac-c554-42d3-bec5-e2381d653dda',
 6: '98dde952-8e91-4825-a73c-a669ed1333dc',
 7: '21bbb306-15ce-4a5a-b523-96ad1f14d03a',
 8: '9b23aba7-0ce7-4cbf-92a5-c4925691ccaf',
 9: '13b8325d-b83c-44f8-9e0d-e017a7619340',
 10: 'dd7dd80f-238f-4b77-b7cf-f5e306a3682c',
 11: 'a7cd4d13-8f0d-465a-ae8b-ccfd30d13d33',
 12: '39cfd7f3-6c3d-4338-b9ae-45fad98b69ab',
 13: '9a79dda3-2377-40dd-9148-660ae7870174',
 14: '27a97b54-7404-4f6a-818f-1235cd6ccc4a',
 15: 'ae2308cd-3cfd-4c81-a84e-4906f6028006',
 16: '3e00fa22-bf27-4641-97a3-8cd964dcc5af',
 17: '07c82d72-6e65-4f17-adcd-62dd7f3f3509',
 18: 'e7b798e9-8091-41f8-a44a-840d8a798ee5',
 19: '14d71aa8-e680-4cb6-915b-03a43c132d46',
 20: '602d6985-3199-4913-bd8c-669f0bf12193',
 21: '1ade788b-8636-44fe-a45a-596b6762dccf',
 22: '888abfa8-3dfa-

In [142]:
vector_store.get_by_ids(['895c8b16-b29c-4181-a9c2-ef780b66575b'])

[Document(id='895c8b16-b29c-4181-a9c2-ef780b66575b', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time')]

## Step 2 - Retrieval

In [161]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 10})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C7507B3200>, search_kwargs={'k': 10})

In [162]:
retrieved_docs = retriever.invoke('What is deepmind')
print(retrieved_docs)

[Document(id='944ac9a4-f8ca-45e5-99b3-80ee9fc31268', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

## Step 3 - Augmentation

In [163]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the answer is not present in the transcript, say:
        I don't know from the provided transcript.

        Context: {context}
        Question: {question}
      """,
    input_variables=["context", "question"]
)

In [164]:
question = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs 

[Document(id='944ac9a4-f8ca-45e5-99b3-80ee9fc31268', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

In [165]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key thing that solved intelligence is that ideas i think it's a combination first\n

In [166]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the answer is not present in the transcript, say:\n        I don't know from the provided transcript.\n\n        Context: and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all 

## Step 4 - Generation

In [167]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

In [168]:
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=HF_TOKEN,
    task="conversational",
    temperature=0.2,
    max_new_tokens=512
)

In [169]:
llm = ChatHuggingFace(llm=endpoint)

In [170]:
final_prompt = prompt.invoke({
    "context": context_text,
    "question": question
})

In [171]:
answer = llm.invoke(final_prompt)

print(answer.content)

I don't know from the provided transcript.


## Building a Chain

In [172]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [173]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [174]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [175]:
parallel_chain.invoke('who is Demis')

{'context': "the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get

In [176]:
parser = StrOutputParser()

In [177]:
main_chain = parallel_chain | prompt | llm | parser

In [178]:
main_chain.invoke('Can you summarize the video')

"The video features a conversation with Demis Hassabis, CEO and co-founder of DeepMind. The discussion covers various topics including the nature of AI, its capabilities, and its future. Hassabis reflects on his personal journey with AI, from playing games to designing them and writing AI for games. He also discusses the importance of language in AI and how it can help in demonstrating the system's ability to generalize across tasks. The conversation touches on the challenges of creating AI systems that can understand and explain complex topics, as well as the potential for AI to enhance human understanding of fundamental physics."